:::{admonition} Download
:class: important

Download this notebook: **{nb-download}`handling_composite_bases.ipynb`**!

:::


# Handling Composite Bases

## Structure of Composite Basis

Composite basis, aka objects of type `AdditiveBasis` or `MultiplicativeBasis`, are containers of multiple "atomic" one-dimensional basis, organized in a tree structure. Every time we add or multiplied two bases, they will be stored as attributes of the `AdditiveBasis` or `MultiplicativeBasis` respectively.

In [1]:
import nemos as nmo

# define a composite basis
add = nmo.basis.RaisedCosineLinearEval(5, label="input1") + nmo.basis.BSplineEval(6, label="input2")

# `add` stores the two 1dimensional bases as attributes
print(add)
print(add.basis1)
print(add.basis2)

'(input1 + input2)': AdditiveBasis(
    basis1='input1': RaisedCosineLinearEval(n_basis_funcs=5, width=2.0),
    basis2='input2': BSplineEval(n_basis_funcs=6, order=4),
)
'input1': RaisedCosineLinearEval(n_basis_funcs=5, width=2.0)
'input2': BSplineEval(n_basis_funcs=6, order=4)


Composing even more, will result in more nesting of attributes.

In [2]:
add = add + nmo.basis.MSplineEval(4, label="input3")
print(add)
print(add.basis1.basis1)
print(add.basis1.basis2)
print(add.basis2)

'((input1 + input2) + input3)': AdditiveBasis(
    basis1='(input1 + input2)': AdditiveBasis(
        basis1='input1': RaisedCosineLinearEval(n_basis_funcs=5, width=2.0),
        basis2='input2': BSplineEval(n_basis_funcs=6, order=4),
    ),
    basis2='input3': MSplineEval(n_basis_funcs=4, order=4),
)
'input1': RaisedCosineLinearEval(n_basis_funcs=5, width=2.0)
'input2': BSplineEval(n_basis_funcs=6, order=4)
'input3': MSplineEval(n_basis_funcs=4, order=4)


## Retrieving Basis Components and Their Parameters
In principle, nesting makes the process of retrieving or setting the parameters of individual components quite cumbersome.

In [3]:
# retreive the number of basis funciton for input2 basis
add.basis1.basis2.n_basis_funcs

6

However, if you associated a label to the basis, you can use it to get the corresponding basis element.

In [4]:
add["input2"]

,n_basis_funcs,6
,label,'input2'
,order,4
,bounds,None
,fill_value,nan


And its parameters can be easily accessed.

In [5]:
add["input2"].n_basis_funcs

6

This works for any sub-element, including the one that are composite.

In [6]:
# get input1 + input2
add["(input1 + input2)"]

,label,'(input1 + input2)'
,input1__bounds,None
,input1__fill_value,nan
,input1__label,'input1'
,input1__n_basis_funcs,5
,input1__width,2.0
,input1,"'input1': Rai...=5, width=2.0)"
,input2__bounds,None
,input2__fill_value,nan
,input2__label,'input2'
,input2__n_basis_funcs,6


Note that the label of this composite basis is assigned automatically. You can overwrite that with a custom label.

In [7]:
add["(input1 + input2)"].label = "my_custom_label"
add

,label,'(my_custom_label + input3)'
,input1__bounds,None
,input1__fill_value,nan
,input1__label,'input1'
,input1__n_basis_funcs,5
,input1__width,2.0
,input1,"'input1': Rai...=5, width=2.0)"
,input2__bounds,None
,input2__fill_value,nan
,input2__label,'input2'
,input2__n_basis_funcs,6


A label can be specified at initialization if the composite basis is defined directly.

In [8]:
nmo.basis.AdditiveBasis(
    nmo.basis.BSplineEval(5),
    nmo.basis.MSplineEval(5),
    label="my_custom_label"
)

,label,'my_custom_label'
,BSplineEval__bounds,None
,BSplineEval__fill_value,nan
,BSplineEval__label,'BSplineEval'
,BSplineEval__n_basis_funcs,5
,BSplineEval__order,4
,BSplineEval,"BSplineEval(n...cs=5, order=4)"
,MSplineEval__bounds,None
,MSplineEval__fill_value,nan
,MSplineEval__label,'MSplineEval'
,MSplineEval__n_basis_funcs,5


And if you are asking yourself what happens when two bases with the same label are composed, well, this results in an error.
This guarantees that the labels are always unique and you can always retrieve a basis using its label.

In [9]:
nmo.basis.BSplineEval(5, label="x") + nmo.basis.MSplineEval(5, label="x")

ValueError: All user-provided labels of basis elements must be distinct.
The basis you are composing share the following labels: 'x'.
Please change the labels for one of the elements before composition.

Because we ensure that all basis labels are unique, you can always retrieve a specific basis using its label, even when the composite basis is made up of many individual basis objects.

In [10]:
# add 10 basis
composite_bas = nmo.basis.MSplineEval(4, label="label_0")
for k in range(1, 10):
    composite_bas = composite_bas + nmo.basis.MSplineEval(4, label=f"label_{k}")

# retreive one of them using the label
composite_bas["label_5"]

,n_basis_funcs,4
,label,'label_5'
,order,4
,bounds,None
,fill_value,nan


## Get and Set Composite Basis Parameters

When working with composite bases, often times one wants to re-configurate specific components. Again, the easiest way to achieve this is labeling each element and using the label to retrieve the basis.

In [11]:
# get the basis function parameter
print(add["input2"].n_basis_funcs)

# set a new value for the parameter
add["input2"].n_basis_funcs = 8
print(add["input2"].n_basis_funcs)

6
8


This change is reflected on the composite basis.

In [12]:
# check that the input2 basis has now 8 basis funcs
add

,label,'(my_custom_label + input3)'
,input1__bounds,None
,input1__fill_value,nan
,input1__label,'input1'
,input1__n_basis_funcs,5
,input1__width,2.0
,input1,"'input1': Rai...=5, width=2.0)"
,input2__bounds,None
,input2__fill_value,nan
,input2__label,'input2'
,input2__n_basis_funcs,8


Note that if you don't provide a label, basis class name is used to construct the keys. If the same basis is repeated, the key is disambiguated by appending an extra numerical identifier.

In [13]:
nmo.basis.BSplineEval(10) + nmo.basis.BSplineEval(5)

,label,'(BSplineEval + BSplineEval_1)'
,BSplineEval__bounds,None
,BSplineEval__fill_value,nan
,BSplineEval__label,'BSplineEval'
,BSplineEval__n_basis_funcs,10
,BSplineEval__order,4
,BSplineEval,"BSplineEval(n...s=10, order=4)"
,BSplineEval_1__bounds,None
,BSplineEval_1__fill_value,nan
,BSplineEval_1__label,'BSplineEval_1'
,BSplineEval_1__n_basis_funcs,5


### Modifying Basis Parameters with `get_params` and `set_params`
Another way to get and set the basis parameter is via the `get_params` and `set_params` methods. This is how `scikit-learn` interacts with basis objects, and so enables cross-validation.

The `get_params` method returns a dictionary, containing all the parameters. The dictionary keys start with the basis label, followed by a double underscore and the name of the parameter.

In [14]:
add.get_params()

{'input1__bounds': None,
 'input1__fill_value': nan,
 'input1__label': 'input1',
 'input1__n_basis_funcs': 5,
 'input1__width': 2.0,
 'input1': 'input1': RaisedCosineLinearEval(n_basis_funcs=5, width=2.0),
 'input2__bounds': None,
 'input2__fill_value': nan,
 'input2__label': 'input2',
 'input2__n_basis_funcs': 8,
 'input2__order': 4,
 'input2': 'input2': BSplineEval(n_basis_funcs=8, order=4),
 'my_custom_label__label': 'my_custom_label',
 'my_custom_label': 'my_custom_label': AdditiveBasis(
     basis1='input1': RaisedCosineLinearEval(n_basis_funcs=5, width=2.0),
     basis2='input2': BSplineEval(n_basis_funcs=8, order=4),
 ),
 'input3__bounds': None,
 'input3__fill_value': nan,
 'input3__label': 'input3',
 'input3__n_basis_funcs': 4,
 'input3__order': 4,
 'input3': 'input3': MSplineEval(n_basis_funcs=4, order=4),
 'label': '(my_custom_label + input3)'}

Each of the key can be used as keyword argument to the `set_method` which in turns sets one or more of the parameter values.

In [15]:
add.set_params(input3__order=3, input1__bounds=(-1,1))

,label,'(my_custom_label + input3)'
,input1__bounds,"(-1.0, ...)"
,input1__fill_value,nan
,input1__label,'input1'
,input1__n_basis_funcs,5
,input1__width,2.0
,input1,'input1': Rai...ill_value=nan)
,input2__bounds,None
,input2__fill_value,nan
,input2__label,'input2'
,input2__n_basis_funcs,8


:::{admonition} Grid definition
:class: info

The parameter keys retrieved by `get_params` are the one needed to define a parameter grid when cross-validating your hyper-parameters with scikit-learn. Learn how to cross-validate basis parameters using [pipelines](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) with [this notebook](sklearn-how-to).
:::


As noted above, when labels are not provided, `get_params` retrieves the auto-generated ones.

In [16]:
basis = nmo.basis.BSplineEval(10) + nmo.basis.BSplineEval(5)
basis.get_params()

{'BSplineEval__bounds': None,
 'BSplineEval__fill_value': nan,
 'BSplineEval__label': 'BSplineEval',
 'BSplineEval__n_basis_funcs': 10,
 'BSplineEval__order': 4,
 'BSplineEval': BSplineEval(n_basis_funcs=10, order=4),
 'BSplineEval_1__bounds': None,
 'BSplineEval_1__fill_value': nan,
 'BSplineEval_1__label': 'BSplineEval_1',
 'BSplineEval_1__n_basis_funcs': 5,
 'BSplineEval_1__order': 4,
 'BSplineEval_1': 'BSplineEval_1': BSplineEval(n_basis_funcs=5, order=4),
 'label': '(BSplineEval + BSplineEval_1)'}

Setting the parameters is still possible, but we recommend to always provide informative labels in order to improve code readability.

In [17]:
basis.set_params(BSplineEval_1__n_basis_funcs=12)

,label,'(BSplineEval + BSplineEval_1)'
,BSplineEval__bounds,None
,BSplineEval__fill_value,nan
,BSplineEval__label,'BSplineEval'
,BSplineEval__n_basis_funcs,10
,BSplineEval__order,4
,BSplineEval,"BSplineEval(n...s=10, order=4)"
,BSplineEval_1__bounds,None
,BSplineEval_1__fill_value,nan
,BSplineEval_1__label,'BSplineEval_1'
,BSplineEval_1__n_basis_funcs,12
